# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50654,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:50742,Total threads: 2
Dashboard: http://127.0.0.1:50743/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:50669,


# Values to edit

In [3]:
sample_method = "sampling_2" # sampling_1 sampling_2
method_2_thresholds = [.8, .5] # .95 .97 .98 .99 1.0
max_cloud_covers = [10, 5, 1]#, 0, 10] # percentage

site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
lowtide_search_range_file = data_path / "ELF24505_satellite_day_search_range.csv"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"

In [5]:
for max_cloud_cover in max_cloud_covers:
    for method_2_threshold in method_2_thresholds:
    
        print(f"Max cloud percentage {max_cloud_cover}, Low tide filtering with MNDWI. "
              f"Sampling threshold {method_2_threshold}.")

        for site_name in site_names:

            sampling.sample_site_mndwi(
                site_name=site_name,
                training_labels_file=training_labels_file,
                uav_folder=uav_folder,
                sample_method=sample_method,
                method_2_threshold=method_2_threshold,
                max_cloud_cover=max_cloud_cover,
                filtered=True,
            )

        counts_summary = sampling.site_sample_counts_by_class_mndwi(
            sample_method=sample_method,
            method_2_threshold=method_2_threshold,
            max_cloud_cover=max_cloud_cover,
                filtered=True,
        )

Max cloud percentage 10, Low tide filtering with MNDWI. Sampling threshold 0.8.
Site Ihutai
Site CatlinsLake
Site CatlinsRiverMouth
Site Childrens
Site Duvauchelle
Site Robinsons
Site Takamatua
Site Purau
	WARNING - satellite image for site Purau - either it's not yet downloaded or there is no suitable low cloud / low tide satellite image. Ignoring this site. Try running `get_site_satellite` first if you expect a satellite image.
Site IveyBay_Nov25
Site IveyBay_Feb26
	WARNING - satellite image for site IveyBay_Feb26 - either it's not yet downloaded or there is no suitable low cloud / low tide satellite image. Ignoring this site. Try running `get_site_satellite` first if you expect a satellite image.
Site LeftBank_Nov25
Site LeftBank_Feb26
Site Paremata_Nov25
Site Paremata_Feb26
Site Paremata_Feb25
Site ThePoint_Nov25
Site ThePoint_Feb26
Site Takapuwahia_Nov25
Site Takapuwahia_Feb26
Site IveyBay_ThePoint_LeftBank_Oct24
	Reporting samples summary in folder: C:\Local\repos\seagrass-detect

Task exception was never retrieved
future: <Task finished name='Task-148982' coro=<Client._gather.<locals>.wait() done, defined at C:\Users\pearsonra\.conda\envs\seagrass\Lib\site-packages\distributed\client.py:2367> exception=AllExit()>
Traceback (most recent call last):
  File "C:\Users\pearsonra\.conda\envs\seagrass\Lib\site-packages\distributed\client.py", line 2376, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 